# Sarvam-1 — Hindi Fake News Detector

fine-tuning `sarvamai/sarvam-1` (7B parameter Indian LLM) on CONSTRAINT 2021 hindi fake news.  
using 8-bit quantization + LoRA so it actually fits on a Kaggle T4 GPU.

**what's in this notebook:**
- 8-bit quant via bitsandbytes  
- LoRA adapters on q/k/v/o projection layers (r=16, alpha=32)  
- cosine LR schedule + linear warmup  
- data augmentation on minority class (word swap) to help with imbalance  
- weighted CrossEntropy loss  
- temperature scaling + threshold tuning on validation set  
- saves soft probs + sample IDs for later ensemble with MuRIL  

**note on sample IDs**: augmented copies get sample_id=-1 so they don't bleed into  
the ensemble alignment step (only real original samples have valid IDs).


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "peft", "bitsandbytes", "nlpaug"], check=True)
print("packages ready")


In [ ]:
import os, zipfile, torch, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special  import softmax
from scipy.optimize import minimize_scalar

from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments,
    BitsAndBytesConfig, EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, matthews_corrcoef,
)
from torch.nn import CrossEntropyLoss
import transformers, peft

print(f"transformers={transformers.__version__}  peft={peft.__version__}")
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none — turn on GPU")


In [ ]:
MODEL_NAME       = "sarvamai/sarvam-1"
MAX_LEN          = 128     # sarvam has longer context but 128 covers most posts
BATCH_SIZE       = 8       # 7B model — need small batch even with 8bit
EPOCHS           = 6
LR               = 2e-5
SEED             = 42
OUTPUT_DIR       = "/kaggle/working/sarvam_best_model"
ZIP_PATH         = "/kaggle/working/sarvam_model_download.zip"
FAKE_LABELS      = {"fake"}
USE_AUGMENTATION = True
AUG_COPIES       = 1       # 1 copy of minority class — more caused overfitting

torch.manual_seed(SEED)
np.random.seed(SEED)
print("config set")


In [ ]:
TRAIN_PATH = "/kaggle/input/datasets/sherwinmazarello/sarvam/cleaned_file2.csv"
VAL_PATH   = "/kaggle/input/datasets/sherwinmazarello/sarvam/cleaned_dataset_valid.csv"
TEST_PATH  = "/kaggle/input/datasets/sherwinmazarello/sarvam/cleaned_dataset_test.csv"

def load_csv(path):
    df = pd.read_csv(path)
    df = df[["Post", "Labels Set"]].dropna().copy()
    df["Post"]       = df["Post"].astype(str).str.strip()
    df["Labels Set"] = df["Labels Set"].astype(str)
    df["label"] = df["Labels Set"].apply(
        lambda x: 1 if {t.strip().lower() for t in x.split(",")} & FAKE_LABELS else 0
    )
    # keep original csv row index as sample_id for ensemble alignment
    df = df.reset_index().rename(columns={"index": "sample_id"})
    return df

df_train = load_csv(TRAIN_PATH)
df_val   = load_csv(VAL_PATH)
df_test  = load_csv(TEST_PATH)

print("dataset breakdown:")
for name, df in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    c = df["label"].value_counts().sort_index()
    total = len(df)
    print(f"  {name}: total={total}  real={c.get(0,0)} ({c.get(0,0)/total*100:.1f}%)  fake={c.get(1,0)} ({c.get(1,0)/total*100:.1f}%)")


In [ ]:
# augment minority class — only on train, only real samples (sample_id != -1)
if USE_AUGMENTATION:
    try:
        import nlpaug.augmenter.word as naw
        aug = naw.RandomWordAug(action="swap")

        minority_class = df_train["label"].value_counts().idxmin()
        minority_df    = df_train[df_train["label"] == minority_class].copy()
        print(f"augmenting {len(minority_df)} minority rows × {AUG_COPIES} copies...")

        aug_rows = []
        for _ in range(AUG_COPIES):
            tmp = minority_df.copy()
            tmp["Post"] = aug.augment(minority_df["Post"].tolist())
            tmp["sample_id"] = -1   # mark as augmented — won't be used in ensemble
            aug_rows.append(tmp)

        df_train_aug = pd.concat([df_train] + aug_rows, ignore_index=True)
        df_train_aug = df_train_aug.sample(frac=1, random_state=SEED).reset_index(drop=True)
        c = df_train_aug["label"].value_counts().sort_index()
        print(f"after aug: total={len(df_train_aug)}  real={c.get(0,0)}  fake={c.get(1,0)}")
    except Exception as e:
        print(f"augmentation failed ({e}), using original train")
        df_train_aug = df_train
else:
    df_train_aug = df_train
    print("augmentation skipped")


In [ ]:
counts = df_train_aug["label"].value_counts().sort_index()
class_weights = torch.tensor((counts.sum() / (2 * counts)).values, dtype=torch.float)
print(f"class weights: real={class_weights[0]:.3f}  fake={class_weights[1]:.3f}")


In [ ]:
print("loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize(batch):
    out = tokenizer(
        batch["Post"],
        truncation=True, padding="max_length", max_length=MAX_LEN,
    )
    out["labels"] = [int(x) for x in batch["label"]]
    return out

def make_ds(df):
    ds = Dataset.from_pandas(df[["Post", "label", "sample_id"]], preserve_index=False)
    ds = ds.map(tokenize, batched=True, remove_columns=["Post", "label"])
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels", "sample_id"])
    return ds

print("tokenising splits...")
train_ds      = make_ds(df_train_aug)   # augmented for training
train_ds_eval = make_ds(df_train)       # original only, for eval
val_ds        = make_ds(df_val)
test_ds       = make_ds(df_test)
print(f"train(aug)={len(train_ds)}  train(eval)={len(train_ds_eval)}  val={len(val_ds)}  test={len(test_ds)}")


## Model: Sarvam-1 + 8-bit + LoRA

Loading in 8-bit quantization to reduce GPU memory. Then wrapping with LoRA on the  
attention projection layers — only training ~0.5% of total parameters.

LoRA config: r=16, alpha=32, dropout=0.05, target_modules = q/k/v/o projections.  
Gradient checkpointing enabled to trade compute for memory.


In [ ]:
print("loading sarvam-1 (8-bit)...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    device_map="auto",
    trust_remote_code=True,
    ignore_mismatched_sizes=True,
)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache    = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    task_type="SEQ_CLS",
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none",
))
model.print_trainable_parameters()


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy":  accuracy_score(labels, preds),
        "f1":        f1_score(labels, preds, zero_division=0),
        "macro_f1":  f1_score(labels, preds, average="macro", zero_division=0),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall":    recall_score(labels, preds, zero_division=0),
        "mcc":       matthews_corrcoef(labels, preds),
    }

class WeightedTrainer(Trainer):
    """Custom trainer that uses class-weighted CE loss."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs["labels"]
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
        )
        logits = outputs.logits
        w      = class_weights.to(device=logits.device, dtype=logits.dtype)
        loss   = CrossEntropyLoss(weight=w)(logits, labels)
        return (loss, outputs) if return_outputs else loss


In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

trainer = WeightedTrainer(
    model=model,
    args=TrainingArguments(
        output_dir                  = OUTPUT_DIR,
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE,
        learning_rate               = LR,
        fp16                        = False,
        warmup_ratio                = 0.1,
        weight_decay                = 0.01,
        lr_scheduler_type           = "cosine",
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "f1",
        greater_is_better           = True,
        save_total_limit            = 2,
        logging_steps               = 50,
        report_to                   = "none",
    ),
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print("starting training...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"best model saved to {OUTPUT_DIR}")


## Temperature scaling + threshold tuning

After training, calibrate the model's confidence using temperature scaling on the val set,  
then find the best classification threshold (also on val). Apply both to test.

Why: the raw logits from a quantized LoRA model tend to be overconfident. Temperature  
scaling (dividing logits by T>1) softens the probabilities without retraining anything.


In [ ]:
print("running inference on all 3 splits...")
train_out = trainer.predict(train_ds_eval)   # original train only
val_out   = trainer.predict(val_ds)
test_out  = trainer.predict(test_ds)

train_logits, train_true = train_out.predictions, train_out.label_ids
val_logits,   val_true   = val_out.predictions,   val_out.label_ids
test_logits,  test_true  = test_out.predictions,  test_out.label_ids

# pull sample IDs in trainer iteration order
train_ids = np.array(train_ds_eval["sample_id"])
val_ids   = np.array(val_ds["sample_id"])
test_ids  = np.array(test_ds["sample_id"])

print(f"logit shapes  — train={train_logits.shape}  val={val_logits.shape}  test={test_logits.shape}")
print(f"ID shapes     — train={train_ids.shape}     val={val_ids.shape}     test={test_ids.shape}")


In [ ]:
# temperature scaling: find T that maximises val macro-F1
def scale_and_threshold(logits, T, threshold):
    probs = softmax(logits / T, axis=1)
    return (probs[:, 1] >= threshold).astype(int), probs

def find_best_T(val_logits, val_true):
    def neg_f1(T):
        scaled = softmax(val_logits / T, axis=1)
        preds  = (scaled[:, 1] >= 0.5).astype(int)
        return -f1_score(val_true, preds, average="macro", zero_division=0)
    res = minimize_scalar(neg_f1, bounds=(0.5, 5.0), method="bounded")
    return res.x

T = find_best_T(val_logits, val_true)
print(f"best temperature T = {T:.4f}")

# find best threshold on val (with optimal T already applied)
val_probs_scaled = softmax(val_logits / T, axis=1)
best_t, best_f1 = 0.5, 0.0
for t in np.arange(0.1, 0.9, 0.01):
    preds = (val_probs_scaled[:, 1] >= t).astype(int)
    score = f1_score(val_true, preds, average="macro", zero_division=0)
    if score > best_f1:
        best_f1, best_t = score, t

print(f"best threshold = {best_t:.3f}  (val macro-F1 = {best_f1:.4f})")

# apply to all splits
train_probs = softmax(train_logits / T, axis=1)
val_probs   = softmax(val_logits   / T, axis=1)
test_probs  = softmax(test_logits  / T, axis=1)

train_pred  = (train_probs[:, 1] >= best_t).astype(int)
val_pred    = (val_probs[:,   1] >= best_t).astype(int)
test_pred   = (test_probs[:,  1] >= best_t).astype(int)


In [ ]:
# full metrics for all 3 splits
results = {}
for split_name, y_true, y_pred in [
    ("Train",      train_true, train_pred),
    ("Val",        val_true,   val_pred),
    ("Test",       test_true,  test_pred),
]:
    results[split_name] = {
        "split":     split_name,
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
        "macro_f1":  f1_score(y_true, y_pred, average="macro", zero_division=0),
        "mcc":       matthews_corrcoef(y_true, y_pred),
    }
    r = results[split_name]
    print(f"{split_name}: acc={r['accuracy']:.4f}  f1={r['f1']:.4f}  macro_f1={r['macro_f1']:.4f}  mcc={r['mcc']:.4f}")


In [ ]:
# confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Sarvam-1 — Confusion Matrices (Train / Val / Test)", fontsize=13, y=1.02)

for ax, (name, y_t, y_p) in zip(axes, [
    ("Train", train_true, train_pred),
    ("Val",   val_true,   val_pred),
    ("Test",  test_true,  test_pred),
]):
    cm = confusion_matrix(y_t, y_p)
    sns.heatmap(cm, annot=True, fmt="d", ax=ax,
                xticklabels=["REAL", "FAKE"], yticklabels=["REAL", "FAKE"],
                cmap="Blues", cbar=False)
    r = results[name] if name != "Val" else results["Val"]
    ax.set_title(f"{name}  acc={r['accuracy']:.3f}  F1={r['f1']:.3f}  MCC={r['mcc']:.3f}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.savefig("/kaggle/working/sarvam_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# training curves from trainer state
history = trainer.state.log_history
train_loss = [l["loss"]      for l in history if "loss" in l and "eval_loss" not in l]
val_loss   = [l["eval_loss"] for l in history if "eval_loss" in l]
val_f1     = [l["eval_f1"]   for l in history if "eval_f1" in l]
ep         = range(1, len(val_loss) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ep, train_loss[:len(ep)], marker="o", label="Train Loss")
axes[0].plot(ep, val_loss,             marker="o", label="Val Loss")
axes[0].set(xlabel="Epoch", ylabel="Loss", title="Loss Curve")
axes[0].legend()
axes[1].plot(ep, val_f1, color="green", marker="o", label="Val F1")
axes[1].set(xlabel="Epoch", ylabel="F1", title="Val F1 per Epoch")
axes[1].legend()
plt.tight_layout()
plt.savefig("/kaggle/working/sarvam_training_curves.png", dpi=150)
plt.show()


In [ ]:
# error analysis — a few examples from each quadrant
df_show              = df_test.copy().reset_index(drop=True)
df_show["predicted"] = test_pred

def show_examples(actual, predicted, title, n=3):
    sub = df_show[(df_show["label"] == actual) & (df_show["predicted"] == predicted)]
    lbl = {0: "REAL", 1: "FAKE"}
    print(f"\n── {title}  (actual={lbl[actual]}, predicted={lbl[predicted]}, total={len(sub)})")
    for _, row in sub.head(n).iterrows():
        print(f"  {row['Post'][:300]}")

show_examples(1, 1, "TRUE POSITIVE  — correctly caught FAKE")
show_examples(0, 0, "TRUE NEGATIVE  — correctly passed REAL")
show_examples(0, 1, "FALSE POSITIVE — REAL wrongly flagged as FAKE")
show_examples(1, 0, "FALSE NEGATIVE — FAKE missed as REAL")


## Save probs + IDs for ensemble

In [ ]:
# saving all soft probs and sample IDs
# ensemble notebook uses np.intersect1d(sarvam_ids, muril_ids) to align both models
# to the exact same articles before combining predictions

PROBS_DIR = "/kaggle/working/ensemble_probs"
os.makedirs(PROBS_DIR, exist_ok=True)

np.save(f"{PROBS_DIR}/sarvam_train_probs.npy",  train_probs)
np.save(f"{PROBS_DIR}/sarvam_valid_probs.npy",  val_probs)
np.save(f"{PROBS_DIR}/sarvam_test_probs.npy",   test_probs)
np.save(f"{PROBS_DIR}/sarvam_train_labels.npy", train_true)
np.save(f"{PROBS_DIR}/sarvam_valid_labels.npy", val_true)
np.save(f"{PROBS_DIR}/sarvam_test_labels.npy",  test_true)
np.save(f"{PROBS_DIR}/sarvam_train_ids.npy",    train_ids)
np.save(f"{PROBS_DIR}/sarvam_valid_ids.npy",    val_ids)
np.save(f"{PROBS_DIR}/sarvam_test_ids.npy",     test_ids)

print(f"saved to {PROBS_DIR}/")
print(f"  train probs: {train_probs.shape}  ids: {train_ids.shape}")
print(f"  val   probs: {val_probs.shape}    ids: {val_ids.shape}")
print(f"  test  probs: {test_probs.shape}   ids: {test_ids.shape}")


In [ ]:
# zip everything for download
files_to_zip = []
for root, _, files in os.walk(OUTPUT_DIR):
    for fname in files:
        full = os.path.join(root, fname)
        files_to_zip.append((full, os.path.relpath(full, "/kaggle/working")))

for f in ["/kaggle/working/sarvam_confusion_matrices.png",
          "/kaggle/working/sarvam_training_curves.png"]:
    if os.path.exists(f):
        files_to_zip.append((f, os.path.basename(f)))

for root, _, files in os.walk(PROBS_DIR):
    for fname in files:
        full = os.path.join(root, fname)
        files_to_zip.append((full, os.path.relpath(full, "/kaggle/working")))

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for full, arc in files_to_zip:
        zf.write(full, arc)

size_mb = os.path.getsize(ZIP_PATH) / (1024**2)
print(f"zip ready: {ZIP_PATH}  ({size_mb:.1f} MB)")
print("download from Kaggle Output tab")
